# 04 — Model Comparison (Week 5)

Decision Tree and Random Forest regressors against the Week 4 Linear Regression baseline.

**Fair comparison rules**
- Same features (Week 4 base set), same leakage-safe preprocessing pipeline, same log(ClosePrice) target.
- Hyperparameters are tuned on the **validation month** (the month before the test month, trained on the 18 months before it).
- The **test month** is scored once per model with its chosen settings.

`scripts/run_models.py` runs the same steps end to end.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from idx_avm import config
from idx_avm.pipeline import prepare_dataset
from idx_avm.models import MODEL_GRIDS, fit_and_score, tune_on_validation, feature_importance

X_MONTHS = json.loads((config.PROCESSED_DIR / "split_metadata.json").read_text())["chosen_window_months"]
data = prepare_dataset(week6_features=True)
DF, TEST_MONTH = data.df, data.months[-1]
VAL_MONTH = TEST_MONTH - 1
print(f"{len(DF):,} rows | train window {X_MONTHS} months | validation {VAL_MONTH} | test {TEST_MONTH}")

## 1. Tune each model on the validation month

In [ ]:
best_params, trials = {}, []
for name in MODEL_GRIDS:
    params, t = tune_on_validation(DF, TEST_MONTH, X_MONTHS, name, flags={})
    best_params[name] = params
    trials.append(t)
pd.concat(trials)[["model", "params", "MdAPE", "MAPE", "R2", "fit_seconds"]].round(3)

## 2. Score once on the test month

In [ ]:
results = {name: fit_and_score(DF, TEST_MONTH, X_MONTHS, name, p, flags={}) for name, p in best_params.items()}
table = pd.DataFrame({n: r.metrics | {"fit_seconds": r.fit_seconds} for n, r in results.items()}).T
table["MdAPE vs baseline (pts)"] = table["MdAPE"] - table.loc["Linear Regression", "MdAPE"]
table.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, r) in zip(axes, results.items()):
    t = r.test
    ax.scatter(t[config.TARGET], t["Predicted"], s=2, alpha=0.25)
    lim = [t[config.TARGET].min(), t[config.TARGET].max()]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set(xscale="log", yscale="log", title=f"{name}\nMdAPE {r.metrics['MdAPE']:.1f}%", xlabel="Actual", ylabel="Predicted")
plt.tight_layout(); plt.show()

## 3. Where each model wins: error by price quintile

In [ ]:
from idx_avm.metrics import metrics_by_price_band
bands = pd.DataFrame({n: metrics_by_price_band(r.test[config.TARGET], r.test["Predicted"])["MdAPE"] for n, r in results.items()})
bands.round(2)

## 4. What the Random Forest relies on

In [ ]:
imp = feature_importance(results["Random Forest"])
imp.plot.barh(x="feature", y="importance", figsize=(7, 5), legend=False).invert_yaxis()
plt.title("Random Forest feature importance (impurity)"); plt.show()

## 5. Model behavior (fill in numbers from the run)

| Model | Strengths | Weaknesses |
|---|---|---|
| **Linear Regression** | Fast, interpretable coefficients, stable across months | One global slope per feature; can't capture location × size interactions; worst on luxury homes |
| **Decision Tree** | Captures non-linear effects and interactions automatically; no scaling needed | High variance: one tree's piecewise-constant predictions are jumpy; overfits without depth / leaf limits |
| **Random Forest** | Averages many de-correlated trees → much lower variance; best accuracy here | Slower to train, larger model file, less interpretable; can't extrapolate beyond the training price range |